# RAG desde cero: manuales de lavadora

En este cuaderno construiremos las partes de RAG de forma explícita: PDF → texto → chunks → embeddings → similitud coseno → contexto → Gemini. La meta es entender qué automatizará ChromaDB en el siguiente cuaderno.

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
from laundry_rag.ingestion import chunk_pages, ingest_manuals
from laundry_rag.retrieval import answer_with_gemini, embed_texts, search_manual


## 1. Ingesta: leer antes de indexar

PyMuPDF extrae texto nativo. Cuando una página tiene menos de 40 caracteres, la función renderiza la página y aplica Tesseract OCR. El corpus queda cacheado en `data/processed/corpus_pages.json`; usa `force=True` sólo para reconstruirlo.

In [ ]:
pages = ingest_manuals()
pd.DataFrame([p.__dict__ for p in pages])[['manual', 'page', 'section', 'extraction_method', 'text']].head(12)

In [ ]:
# Confirma que el manual escaneado pasó por OCR y revisa una página antes de seguir.
ocr_pages = [p for p in pages if p.extraction_method == 'ocr']
print(f'Páginas OCR: {len(ocr_pages)}')
print(ocr_pages[0].text[:1_000] if ocr_pages else 'No se detectaron páginas OCR.')

## 2. Chunking propio

Cada chunk conserva fuente, modelo, página y sección. Empezamos con 220 palabras y 40 de solape; son variables que se deben experimentar, no valores universales.

In [ ]:
chunks = chunk_pages(pages, chunk_words=220, overlap_words=40)
pd.DataFrame([{'id': c.id, **c.metadata(), 'palabras': len(c.text.split())} for c in chunks]).head()

## 3. Embeddings y recuperación manual

`embed_texts` usa un modelo multilingüe local. La similitud coseno sí se calcula en nuestro código, y los mejores fragmentos se inspeccionan antes de pedir una respuesta al LLM.

In [ ]:
embeddings = embed_texts([chunk.text for chunk in chunks])
embeddings.shape

In [ ]:
pregunta = '¿Cómo debo cuidar la lavadora si no la usaré durante vacaciones?'
evidencia = search_manual(pregunta, chunks, embeddings, top_k=4)

for item in evidencia:
    print(f'[{item.manual}, p. {item.page}] distancia={item.distance:.3f} · {item.section}')
    print(item.text[:500], '\n')


## 4. Generación fundamentada

Gemini recibe exclusivamente los chunks recuperados. La instrucción exige citas y una abstención literal si el contexto no basta. Define `GOOGLE_API_KEY` en `.env` antes de ejecutar la siguiente celda.

In [ ]:
if os.getenv('GOOGLE_API_KEY'):
    respuesta = answer_with_gemini(pregunta, evidencia)
    print(respuesta)
else:
    print('Define GOOGLE_API_KEY en .env para ejecutar la generación con Gemini.')


## Ejercicios

1. Cambia `chunk_words` a 100 y a 400; compara qué páginas aparecen.
2. Prueba preguntas de seguridad, instalación, ciclos, mantenimiento y solución de problemas.
3. Pregunta: *¿Cuál es la receta para preparar pan de masa madre?* La respuesta debe abstenerse. Si no lo hace, inspecciona primero los chunks recuperados, no sólo el prompt.